# 02 - 向量数据库 (Vector Databases)

## 学习目标

本notebook深入讲解向量数据库的核心技术：

1. **相似度度量**
   - 余弦相似度
   - 欧氏距离
   - 点积相似度

2. **向量存储**
   - 文档管理
   - 向量索引
   - CRUD操作

3. **相似度搜索**
   - Top-K检索
   - 阈值过滤
   - 性能优化

4. **持久化**
   - 保存与加载
   - 数据格式

---

## 0. 环境设置

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import json
from pathlib import Path

from embeddings import DenseEmbedding, EmbeddingConfig
from vector_store import Document, SimpleVectorStore, SearchResult

print("环境导入完成！")

## 1. 相似度度量

### 1.1 三种度量方式对比

In [ ]:
def cosine_similarity(a, b):
    """余弦相似度: 关注方向，忽略模长"""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

def euclidean_distance(a, b):
    """欧氏距离: 绝对距离"""
    return np.linalg.norm(a - b)

def dot_product(a, b):
    """点积: 高效，需归一化"""
    return np.dot(a, b)

# 测试向量
v1 = np.array([1.0, 2.0, 3.0])
v2 = np.array([1.0, 2.0, 3.0])  # 相同向量
v3 = np.array([2.0, 4.0, 6.0])  # 成比例
v4 = np.array([-1.0, -2.0, -3.0])  # 相反方向

vectors = [("v1 vs v2", v1, v2), ("v1 vs v3", v1, v3), ("v1 vs v4", v1, v4)]

print("相似度度量对比:\n")
print(f"{'对比':>12} {'余弦':>10} {'欧氏':>10} {'点积':>10}")
print("-" * 46)

for name, a, b in vectors:
    cos_sim = cosine_similarity(a, b)
    euc_dist = euclidean_distance(a, b)
    dot_prod = dot_product(a, b)
    print(f"{name:>12} {cos_sim:>10.4f} {euc_dist:>10.4f} {dot_prod:>10.4f}")

## 2. 文档管理

In [ ]:
# 创建文档
documents = [
    Document(
        content="机器学习是人工智能的核心技术，通过数据训练模型",
        metadata={"source": "ai_intro.txt", "page": 1, "category": "AI"}
    ),
    Document(
        content="深度学习使用多层神经网络进行特征学习",
        metadata={"source": "deep_learning.txt", "page": 2, "category": "DL"}
    ),
    Document(
        content="自然语言处理让计算机理解人类语言",
        metadata={"source": "nlp_intro.txt", "page": 1, "category": "NLP"}
    ),
    Document(
        content="计算机视觉使机器能够识别图像和视频",
        metadata={"source": "cv_intro.txt", "page": 1, "category": "CV"}
    ),
    Document(
        content="强化学习通过奖励信号学习最优策略",
        metadata={"source": "rl_intro.txt", "page": 1, "category": "RL"}
    ),
]

print(f"创建了 {len(documents)} 个文档")
for doc in documents:
    print(f"\n[{doc.doc_id[:8]}...] {doc.content[:40]}...")
    print(f"  元数据: {doc.metadata}")

## 3. 向量存储

In [ ]:
# 初始化嵌入模型和向量存储
config = EmbeddingConfig(dimension=128, normalize=True)
embedding = DenseEmbedding(config, random_seed=42)

# 为文档生成嵌入
for doc in documents:
    doc.embedding = embedding.embed_text(doc.content)

# 创建向量存储
store = SimpleVectorStore(metric="cosine")

# 添加文档
doc_ids = store.add_documents(documents)

print(f"向量存储信息:")
print(f"  文档数量: {store.count}")
print(f"  距离度量: cosine")
print(f"  向量维度: {config.dimension}")

## 4. 相似度搜索

In [ ]:
def search_and_display(query: str, store: SimpleVectorStore, 
                       embedding: DenseEmbedding, top_k: int = 3):
    """搜索并显示结果。"""
    query_vec = embedding.embed_text(query)
    results = store.search(query_vec, top_k=top_k)
    
    print(f"\n查询: {query}")
    print(f"\nTop-{len(results)} 相关文档:")
    print("-" * 60)
    
    for r in results:
        print(f"\n排名: {r.rank}")
        print(f"分数: {r.score:.4f}")
        print(f"内容: {r.document.content}")
        print(f"来源: {r.document.metadata.get('source', 'N/A')}")
    
    return results

# 测试查询
queries = [
    "什么是深度学习",
    "人工智能和机器学习",
    "图像识别技术",
]

for query in queries:
    search_and_display(query, store, embedding, top_k=2)

## 5. 不同距离度量对比

In [ ]:
query = "深度学习和神经网络"
query_vec = embedding.embed_text(query)

metrics = ["cosine", "euclidean", "dot"]

print(f"查询: {query}\n")
print(f"{'度量':>12} {'Top-1文档':<40} {'分数':>10}\n")
print("-" * 70)

for metric in metrics:
    s = SimpleVectorStore(metric=metric)
    s.add_documents(documents)
    results = s.search(query_vec, top_k=1)
    
    if results:
        r = results[0]
        print(f"{metric:>12} {r.document.content[:40]:<40} {r.score:>10.4f}")

## 6. CRUD操作

In [ ]:
print(f"初始文档数: {store.count}")

# 获取文档
doc_id = doc_ids[0]
doc = store.get(doc_id)
print(f"\n获取文档 [{doc_id[:8]}...]:")
print(f"  内容: {doc.content}")

# 删除文档
deleted_count = store.delete([doc_ids[0]])
print(f"\n删除文档数: {deleted_count}")
print(f"剩余文档数: {store.count}")

# 重新添加
store.add_documents([doc])
print(f"\n重新添加后文档数: {store.count}")

## 7. 持久化

In [ ]:
# 保存到文件
save_path = "/tmp/vector_store.json"
store.save(save_path)
print(f"向量存储已保存到: {save_path}")

# 查看文件内容
with open(save_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"\n保存的数据结构:")
print(f"  距离度量: {data['metric']}")
print(f"  文档数量: {len(data['documents'])}")
print(f"  向量数量: {len(data['vectors'])}")

In [ ]:
# 从文件加载
loaded_store = SimpleVectorStore.load(save_path)
print(f"\n从文件加载的向量存储:")
print(f"  文档数: {loaded_store.count}")
print(f"  度量: {loaded_store.metric}")

# 验证搜索功能
results = loaded_store.search(query_vec, top_k=2)
print(f"\n搜索结果数: {len(results)}")

## 8. 性能测试

In [ ]:
import time

def benchmark_search(num_docs_list: list, num_queries: int = 10):
    """测试不同文档数量的搜索性能。"""
    results = []
    
    for num_docs in num_docs_list:
        # 创建随机文档
        docs = []
        for i in range(num_docs):
            content = f"测试文档{i}，包含一些随机内容用于性能测试"
            doc = Document(content=content, metadata={"index": i})
            doc.embedding = embedding.embed_text(content)
            docs.append(doc)
        
        # 添加到存储
        s = SimpleVectorStore(metric="cosine")
        s.add_documents(docs)
        
        # 测试查询
        query_vec = embedding.embed_text("测试查询")
        
        start = time.time()
        for _ in range(num_queries):
            _ = s.search(query_vec, top_k=5)
        elapsed = time.time() - start
        
        avg_time = elapsed / num_queries * 1000  # ms
        results.append((num_docs, avg_time))
        print(f"文档数: {num_docs:5d}, 平均查询时间: {avg_time:.3f} ms")
    
    return results

# 性能测试
num_docs_list = [100, 500, 1000, 2000]
print("性能测试结果:\n")
benchmark_search(num_docs_list)

## 9. 总结

本notebook涵盖了向量数据库的核心功能：

1. **相似度度量**: 余弦/欧氏/点积三种方式
2. **文档管理**: CRUD操作完整实现
3. **相似度搜索**: Top-K检索和阈值过滤
4. **持久化**: JSON格式保存和加载
5. **性能测试**: 大规模数据下的查询效率

### 关键要点

- **余弦相似度**: 最常用的文本检索度量
- **向量归一化**: 点积等价于余弦相似度
- **持久化**: 支持保存和加载向量索引
- **性能优化**: 批量操作和向量化计算